Carlos Matias Jose Silva Timmermann

22.681.769-7

In [10]:
import os
import pickle
from pathlib import Path
import numpy as np

Codigo :

In [11]:

#Clase para que leer, limpiar, analizar, exportar e inspeccionar "CatalogoEstelar"

class CatalogoEstelar: 

    #Def para el constructor de la clase

        def __init__(self, ruta_csv):

        #Carga el archivo CSV, almacena las columnas como arreglos independientes y ejecuta la limpieza de datos

            self.ruta_csv = Path(ruta_csv)

        #Carga los datos del archivo CSV utilizando numpy.genfromtxt y ajusta el dtype segun el archivo

            datos = np.genfromtxt(self.ruta_csv, delimiter = ",", dtype = None, encoding = "utf-8", skip_header = 1)

        #Encabezados o nombres de las columnas, si el CSV tiene otra cantidad o orden, ajustamos segun corresponda

            self.id = datos["f0"] if datos.dtype.names else datos[:, 0]
            self.nombre = datos["f1"] if datos.dtype.names else datos[:, 1]
            self.temperatura = np.array(datos["f2"] if datos.dtype.names else datos[:, 2], dtype=float)
            self.luminosidad = np.array(datos["f3"] if datos.dtype.names else datos[:, 3], dtype=float)
            self.distancia = np.array(datos["f4"] if datos.dtype.names else datos[:, 4], dtype=float)

        #Se convierte el id y nombre a arreglos NumPy de cadenas estrictas

            self.id = np.array(self.id, dtype=str)
            self.nombre = np.array(self.nombre, dtype=str)

        #Se ejecuta la limpieza automática inmediatamente

            self.limpiar_datos()

    #Def para la limpieza de datos

        def limpiar_datos(self):

            #Mascara para valores finitos en columnas numericas

                mascara_finitos = (np.isfinite(self.temperatura) & np.isfinite(self.luminosidad) & np.isfinite(self.distancia))

            #Mascara para propiedades fisicas estrictamente positivas

                mascara_positivos = ((self.temperatura > 0) & (self.luminosidad > 0) & (self.distancia > 0))

            #Mascara para combinar validez numerica y fisica

                mascara_valida = mascara_finitos & mascara_positivos

            #Se aplica filtrado inicial de validez

                id_temperatura = self.id[mascara_valida]
                nombre_temperatura = self.nombre[mascara_valida]
                temperatura_temperarura = self.temperatura[mascara_valida]
                luminosidad_temperatura = self.luminosidad[mascara_valida]
                distancia_temperatura = self.distancia[mascara_valida]

            #Identifica duplicados conservando solo la primera aparicion

                _,indices_unicos = np.unique(id_temperatura, return_index = True)

            #np.unique devuelve índices desordenados, los ordenamos para preservar secuencia original

                indices_unicos = np.sort(indices_unicos)

            #Aplica cambios a todas las columnas

                self.id = id_temperatura[indices_unicos]
                self.nombre = nombre_temperatura[indices_unicos]
                self.temperatura = temperatura_temperarura[indices_unicos]
                self.luminosidad = luminosidad_temperatura[indices_unicos]
                self.distancia = distancia_temperatura[indices_unicos]

    #Def para el resumen de estadisticas
    
        def resumen(self):
    
            #Se calcula estadisticas del catalogo estelar
    
                numero_estrellas = len(self.id)
                temperatura_media = np.mean(self.temperatura)
                temperatura_maxima = np.max(self.temperatura)
                luminosidad_mediana = np.median(self.luminosidad)
                distancia_minima = np.min(self.distancia)
                distancia_maxima = np.max(self.distancia)
    
            #Identificacion por indices de condiciones extremas
    
                indice_mas_caliente = np.argmax(self.temperatura)
                indice_mas_cercana = np.argmin(self.distancia)
                nombre_mas_caliente = self.nombre[indice_mas_caliente]
                nombre_mas_cercana = self.nombre[indice_mas_cercana]
    
            #Printeo de resultados
    
                print("Resumen del catalogo estelar limpio")
                print(f"Numero total de estrellas : {numero_estrellas}")
                print(f"Temperatura media : {temperatura_media:.2f} K")
                print(f"Temperatura maxima  {temperatura_maxima:.2f} K")
                print(f"Luminosidad mediana : {luminosidad_mediana:.4e} L_sol")
                print(f"Distancia minima : {distancia_minima:.2f} pc")
                print(f"Distancia maxima : {distancia_maxima:.2f} pc")
                print(f"Estrella con mayor temperatura : {nombre_mas_caliente} ({temperatura_maxima:.2f} K)")
                print(f"Estrella mas cercana : {nombre_mas_cercana} ({distancia_minima:.2f} pc)")

    #Def para preparar verificar y crear un escritorio de salida si no existe

        def preparar_directorio(self, ruta = "resultados"):
                p = Path(ruta)
                p.mkdir(parents = True, exist_ok = True)
                return p

    #Def para guardar todas las columnas en formato CSV con encabezados
    
        def guardar_csv(self, ruta = "resultados/catalogo_copia.csv"):
                self.preparar_directorio(Path(ruta).parent)
                header = "id, nombre, temperatura, luminosidad, distancia"
                datos_completos = np.column_stack(
                    (self.id, self.nombre, self.temperatura, self.luminosidad, self.distancia)
                )
                np.savetxt(ruta, datos_completos, delimiter = ",", fmt = "%s", header = header, comments = "")

    
    #Def para guardar solo la columna de temperaturas en formato binario .npy

        def guardar_npy(self, ruta = "resultados/temperaturas.npy"):
                self.preparar_directorio(Path(ruta).parent)
                np.save(ruta, self.temperatura)

    #Def para cargar y retornar los datos de un archivo .npy

        def cargar_npy(self, ruta="resultados/temperaturas.npy"):
                return np.load(ruta)


    #Def para guardar todas las columnas en un archivo comprimido .npz sin necesidad de pickle
    
        def guardar_npz(self, ruta = "resultados/catalogo_completo.npz"):
                self.preparar_directorio(Path(ruta).parent)
                np.savez_compressed(ruta, id = self.id, nombre = self.nombre, temperatura = self.temperatura, luminosidad = self.luminosidad, distancia = self.distancia)

    #Def para cargar y retornar el contenido de un archivo .npz
    
        def cargar_npz(self, ruta = "resultados/catalogo_completo.npz"):
                return np.load(ruta)

    #Def para guardar un diccionario con todas las columnas utilizando pickle
    
        def guardar_pickle(self, ruta = "resultados/catalogo_completo.pkl"):
            self.preparar_directorio(Path(ruta).parent)
            diccionario_datos = {
                "id" : self.id,
                "nombre" : self.nombre,
                "temperatura" : self.temperatura,
                "luminosidad" : self.luminosidad,
                "distancia" : self.distancia,
            }
            with open(ruta, "wb") as f:
                pickle.dump(diccionario_datos, f)

    #Def para cargar y retornar los datos guardados en un archivo Pickle
    
        def cargar_pickle(self, ruta="resultados/catalogo_completo.pkl"):
            with open(ruta, "rb") as f:
                return pickle.load(f)

    #Def para verificar la integridad de los datos guardados en CSV, NPY, NPZ y Pickle
    
        def verificar_archivos(self):
            print("Verificacion de integridad de archivos generados : ")
    
        #Verificacion NPY (solo temperatura)
    
            temperatura_npy = self.cargar_npy()
            npy_ok = (len(temperatura_npy) == len(self.temperatura)) and np.allclose(temperatura_npy, self.temperatura)
            print(f"Archivo NPY (temperaturas) : {'Paso' if npy_ok else 'Fallo'}")
    
        #Verificacion NPZ (catalogo completo)
    
            npz = self.cargar_npz()
            npz_ok = (np.array_equal(npz["id"], self.id) 
                      and np.array_equal(npz["nombre"], self.nombre) 
                      and np.allclose(npz["temperatura"], self.temperatura) 
                      and np.allclose(npz["luminosidad"], self.luminosidad) 
                      and np.allclose(npz["distancia"], self.distancia)
                    )
            print(f"Archivo NPZ (completo) : {'Paso' if npz_ok else 'Fallo'}")
    
        #Verificacion Pickle (catalogo completo)
    
            pkl = self.cargar_pickle()
            pkl_ok = (
                np.array_equal(pkl["id"], self.id)
                and np.array_equal(pkl["nombre"], self.nombre)
                and np.allclose(pkl["temperatura"], self.temperatura)
                and np.allclose(pkl["luminosidad"], self.luminosidad)
                and np.allclose(pkl["distancia"], self.distancia)
            )
            print(f"Archivo Pickle (completo) : {'Paso' if pkl_ok else 'Fallo'}")
    
        #Verificacion CSV (releyendo el archivo)
    
            cat_copia = CatalogoEstelar("resultados/catalogo_copia.csv")
            csv_ok = (
                np.array_equal(cat_copia.id, self.id)
                and np.array_equal(cat_copia.nombre, self.nombre)
                and np.allclose(cat_copia.temperatura, self.temperatura)
                and np.allclose(cat_copia.luminosidad, self.luminosidad)
                and np.allclose(cat_copia.distancia, self.distancia)
            )
            print(f"Archivo CSV (copia releida) : {'Paso' if csv_ok else 'Fallo'}")

    #Def para medir y presentar en una tabla el tamaño en bytes/KB de cada formato generado
    
        def comparar_formatos(self):
            archivos = {
                "CSV (catalogo_copia.csv)": "resultados/catalogo_copia.csv",
                "NPY (temperaturas.npy)": "resultados/temperaturas.npy",
                "NPZ (catalogo_completo.npz)": "resultados/catalogo_completo.npz",
                "Pickle (catalogo_completo.pkl)": "resultados/catalogo_completo.pkl",
            }
            print(f"{'Formato':<32} | {'Tamaño (Bytes)':<15} | {'Tamaño (KB)':<12}")
            for nombre, ruta in archivos.items():
                if os.path.exists(ruta):
                    size_bytes = os.path.getsize(ruta)
                    size_kb = size_bytes / 1024
                    print(f"{nombre:<32} | {size_bytes:<15} | {size_kb:<12.2f}")
                else:
                    print(f"{nombre:<32} | Archivo no encontrado")

catalogo = CatalogoEstelar("catalogo_estelar.csv")
catalogo.resumen()
catalogo.guardar_csv()
catalogo.guardar_npy()
catalogo.guardar_npz()
catalogo.guardar_pickle()
catalogo.verificar_archivos()
catalogo.comparar_formatos()

Resumen del catalogo estelar limpio
Numero total de estrellas : 92
Temperatura media : 6388.60 K
Temperatura maxima  9829.36 K
Luminosidad mediana : 6.0274e+00 L_sol
Distancia minima : 58.03 pc
Distancia maxima : 490.81 pc
Estrella con mayor temperatura : Estrella_0006 (9829.36 K)
Estrella mas cercana : Estrella_0039 (58.03 pc)
Verificacion de integridad de archivos generados : 
Archivo NPY (temperaturas) : Paso
Archivo NPZ (completo) : Paso
Archivo Pickle (completo) : Paso
Archivo CSV (copia releida) : Paso
Formato                          | Tamaño (Bytes)  | Tamaño (KB) 
CSV (catalogo_copia.csv)         | 7040            | 6.88        
NPY (temperaturas.npy)           | 864             | 0.84        
NPZ (catalogo_completo.npz)      | 3769            | 3.68        
Pickle (catalogo_completo.pkl)   | 9174            | 8.96        
